# Day 4: Data Wrangling & System Integration
## Stage 1: Fundamental Data Science & Analytics — Epson Training

**Learning Objectives:**
- Handle missing values and outliers systematically
- Use SQL to query production databases
- Merge sensor (IoT) data with ERP data for integrated analysis
- Build a complete data pipeline from raw sources to insights

> 💡 **Google Colab Tip:** SQLite is available in Python's standard library — no extra installation needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime, timedelta

sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)

print("✅ Libraries loaded")

---
## Part 1: Advanced Missing Value Handling

In [ ]:
# Generate production sensor dataset with realistic missing patterns
n = 400
start = datetime(2024, 1, 1)
machines = ["M001", "M002", "M003", "M004"]

df_raw = pd.DataFrame({
    "record_id": range(1, n + 1),
    "timestamp": [start + timedelta(hours=i) for i in range(n)],
    "machine_id": np.tile(machines, n // len(machines)),
    "temperature_c": np.round(np.random.normal(72, 3, n), 2),
    "vibration_mms": np.round(np.abs(np.random.normal(3, 1.2, n)), 3),
    "pressure_hpa": np.round(np.random.normal(1013, 5, n), 1),
    "rpm": np.random.randint(1400, 1600, n).astype(float),
    "production_units": np.random.randint(180, 220, n).astype(float),
    "energy_kwh": np.round(np.random.normal(15, 2, n), 2),
})

# Simulate missing data patterns:
# MCAR — completely random
mcar_idx = np.random.choice(n, 20, replace=False)
df_raw.loc[mcar_idx, "temperature_c"] = np.nan

# MAR — missing when machine is M003 (sensor fault)
m003_idx = df_raw[df_raw["machine_id"] == "M003"].sample(15).index
df_raw.loc[m003_idx, "vibration_mms"] = np.nan

# MNAR — pressure missing at extreme values
extreme_idx = df_raw[df_raw["pressure_hpa"] > 1022].sample(10).index
df_raw.loc[extreme_idx, "pressure_hpa"] = np.nan

# Some rpm and production data missing
df_raw.loc[np.random.choice(n, 8, replace=False), "rpm"] = np.nan
df_raw.loc[np.random.choice(n, 5, replace=False), "production_units"] = np.nan

print("=== Raw Data: Missing Value Report ===")
missing_report = pd.DataFrame({
    "Missing": df_raw.isnull().sum(),
    "Missing %": (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    "Type": ["N/A", "N/A", "N/A", "MCAR", "MAR (M003)", "MNAR (high P)", "Random", "Random", "N/A"]
})
print(missing_report[missing_report["Missing"] > 0])

In [ ]:
# --- Imputation Strategies ---
df_clean = df_raw.copy()

# Strategy 1: Machine-specific median for temperature (MCAR)
df_clean["temperature_c"] = df_clean.groupby("machine_id")["temperature_c"].transform(
    lambda x: x.fillna(x.median())
)
print("✅ temperature_c: filled with machine-specific median")

# Strategy 2: Forward-fill then backward-fill for vibration (MAR)
df_clean["vibration_mms"] = (
    df_clean.groupby("machine_id")["vibration_mms"]
    .transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))
)
print("✅ vibration_mms: filled with ffill/bfill per machine")

# Strategy 3: Global median for pressure (MNAR — note the limitation)
df_clean["pressure_hpa"] = df_clean["pressure_hpa"].fillna(df_clean["pressure_hpa"].median())
print("✅ pressure_hpa: filled with global median (flag: MNAR values may be biased)")

# Strategy 4: Linear interpolation for continuous time-series (rpm)
df_clean["rpm"] = df_clean["rpm"].interpolate(method="linear")
print("✅ rpm: filled with linear interpolation")

# Strategy 5: Drop rows missing production_units (critical metric)
before = len(df_clean)
df_clean = df_clean.dropna(subset=["production_units"])
print(f"✅ production_units: dropped {before - len(df_clean)} rows with missing values")

print(f"\nFinal missing values: {df_clean.isnull().sum().sum()}")
print(f"Dataset size: {len(df_clean)} rows")

In [ ]:
# Visualize before/after imputation for temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before: show missing gaps
axes[0].plot(df_raw["timestamp"][:100], df_raw["temperature_c"][:100],
             color="steelblue", linewidth=1)
missing_mask = df_raw["temperature_c"][:100].isnull()
axes[0].scatter(df_raw["timestamp"][:100][missing_mask],
                [72] * missing_mask.sum(), color="red", s=50, zorder=5,
                label=f"Missing ({missing_mask.sum()} pts)")
axes[0].set_title("Temperature — Before Imputation (first 100 records)")
axes[0].set_ylabel("°C")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=25)

# After: filled
axes[1].plot(df_clean["timestamp"][:100], df_clean["temperature_c"][:100],
             color="seagreen", linewidth=1)
axes[1].set_title("Temperature — After Imputation (first 100 records)")
axes[1].set_ylabel("°C")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig("day4_imputation_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Part 2: Outlier Detection & Treatment

In [ ]:
# Inject clear outliers into the clean data
df_with_outliers = df_clean.copy()
outlier_idx = np.random.choice(len(df_with_outliers), 15, replace=False)
df_with_outliers.iloc[outlier_idx[:8], df_with_outliers.columns.get_loc("temperature_c")] = \
    np.random.uniform(95, 105, 8)
df_with_outliers.iloc[outlier_idx[8:], df_with_outliers.columns.get_loc("vibration_mms")] = \
    np.random.uniform(10, 15, 7)

def outlier_summary(df, col):
    """Report outliers using both IQR and Z-score methods."""
    series = df[col].dropna()

    # IQR method
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    iqr_mask = (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)

    # Z-score method
    z_scores = np.abs((series - series.mean()) / series.std())
    z_mask = z_scores > 3

    print(f"  {col}:")
    print(f"    IQR method:     {iqr_mask.sum()} outliers")
    print(f"    Z-score method: {z_mask.sum()} outliers")
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print("=== Outlier Detection ===")
bounds_temp = outlier_summary(df_with_outliers, "temperature_c")
bounds_vib = outlier_summary(df_with_outliers, "vibration_mms")

# Treatment: Winsorization (cap at bounds)
df_treated = df_with_outliers.copy()
df_treated["temperature_c"] = df_treated["temperature_c"].clip(*bounds_temp)
df_treated["vibration_mms"] = df_treated["vibration_mms"].clip(*bounds_vib)

print("\n✅ Outliers treated with Winsorization")
print(f"  Temp max before: {df_with_outliers['temperature_c'].max():.1f}, "
      f"after: {df_treated['temperature_c'].max():.1f}")

In [ ]:
# Visualize outlier treatment
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Outlier Detection & Treatment", fontsize=13, fontweight="bold")

# Before boxplots
axes[0, 0].boxplot(df_with_outliers["temperature_c"].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor="tomato", alpha=0.6))
axes[0, 0].set_title("Temperature — Before Treatment")
axes[0, 0].set_ylabel("°C")

axes[0, 1].boxplot(df_with_outliers["vibration_mms"].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor="tomato", alpha=0.6))
axes[0, 1].set_title("Vibration — Before Treatment")
axes[0, 1].set_ylabel("mm/s")

# After boxplots
axes[1, 0].boxplot(df_treated["temperature_c"].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor="seagreen", alpha=0.6))
axes[1, 0].set_title("Temperature — After Treatment")
axes[1, 0].set_ylabel("°C")

axes[1, 1].boxplot(df_treated["vibration_mms"].dropna(), vert=True, patch_artist=True,
                   boxprops=dict(facecolor="seagreen", alpha=0.6))
axes[1, 1].set_title("Vibration — After Treatment")
axes[1, 1].set_ylabel("mm/s")

plt.tight_layout()
plt.savefig("day4_outlier_treatment.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Part 3: SQL for Querying Production Databases

In [ ]:
# --- Build an In-Memory SQLite Production Database ---

conn = sqlite3.connect(":memory:")

# Prepare and load tables into SQLite
df_sensors = df_treated[["record_id", "timestamp", "machine_id",
                          "temperature_c", "vibration_mms",
                          "pressure_hpa", "rpm", "energy_kwh"]].copy()
df_sensors["timestamp"] = df_sensors["timestamp"].astype(str)
df_sensors.to_sql("sensor_data", conn, index=False, if_exists="replace")

# ERP/Production table
product_types = ["Printer_A4", "Printer_A3", "Scanner_Pro", "Projector_HD"]
df_production = df_treated[["record_id", "timestamp", "machine_id", "production_units"]].copy()
df_production["timestamp"] = df_production["timestamp"].astype(str)
df_production["product_type"] = np.random.choice(product_types, len(df_production))
df_production["defect_count"] = np.random.randint(0, 8, len(df_production))
df_production["shift"] = df_production["timestamp"].apply(
    lambda t: "Day" if 6 <= int(t[11:13]) < 18 else "Night"
)
df_production.to_sql("production", conn, index=False, if_exists="replace")

# Machine master table
df_machines = pd.DataFrame({
    "machine_id": machines,
    "machine_name": ["CNC Line A", "CNC Line B", "Assembly Line C", "Inspection Line D"],
    "machine_type": ["CNC", "CNC", "Assembly", "Inspection"],
    "location": ["Building 1", "Building 1", "Building 2", "Building 2"],
    "install_year": [2018, 2019, 2020, 2021],
})
df_machines.to_sql("machines", conn, index=False, if_exists="replace")

print("✅ SQLite database created with 3 tables:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0, 0]
    print(f"   {t}: {count:,} rows")

In [ ]:
# --- SQL Query 1: Basic SELECT ---
print("=== Query 1: Recent Sensor Readings ===")
q1 = """
SELECT
    s.record_id,
    s.timestamp,
    s.machine_id,
    m.machine_name,
    s.temperature_c,
    s.vibration_mms,
    s.energy_kwh
FROM sensor_data s
JOIN machines m ON s.machine_id = m.machine_id
ORDER BY s.timestamp DESC
LIMIT 10
"""
result1 = pd.read_sql(q1, conn)
print(result1.to_string(index=False))

In [ ]:
# --- SQL Query 2: Aggregation & GROUP BY ---
print("=== Query 2: Machine Performance Summary ===")
q2 = """
SELECT
    s.machine_id,
    m.machine_name,
    m.machine_type,
    COUNT(*)                             AS total_records,
    ROUND(AVG(s.temperature_c), 2)       AS avg_temp_c,
    ROUND(MAX(s.temperature_c), 2)       AS max_temp_c,
    ROUND(AVG(s.vibration_mms), 3)       AS avg_vibration,
    ROUND(SUM(s.energy_kwh), 1)          AS total_energy_kwh
FROM sensor_data s
JOIN machines m ON s.machine_id = m.machine_id
GROUP BY s.machine_id, m.machine_name, m.machine_type
ORDER BY avg_temp_c DESC
"""
result2 = pd.read_sql(q2, conn)
print(result2.to_string(index=False))

In [ ]:
# --- SQL Query 3: Filtering with WHERE & HAVING ---
print("=== Query 3: High-Temperature Alert Periods ===")
q3 = """
SELECT
    machine_id,
    COUNT(*) AS high_temp_events,
    ROUND(AVG(temperature_c), 2) AS avg_high_temp,
    ROUND(MAX(temperature_c), 2) AS peak_temp
FROM sensor_data
WHERE temperature_c > 77
GROUP BY machine_id
HAVING high_temp_events >= 3
ORDER BY high_temp_events DESC
"""
result3 = pd.read_sql(q3, conn)
print(result3.to_string(index=False))

In [ ]:
# --- SQL Query 4: Production Quality by Shift ---
print("=== Query 4: Quality by Shift and Machine ===")
q4 = """
SELECT
    p.machine_id,
    p.shift,
    COUNT(*)                                      AS batches,
    ROUND(SUM(p.production_units), 0)             AS total_units,
    ROUND(AVG(p.production_units), 1)             AS avg_units_per_batch,
    ROUND(SUM(p.defect_count) * 100.0
          / SUM(p.production_units), 2)           AS defect_rate_pct
FROM production p
GROUP BY p.machine_id, p.shift
ORDER BY p.machine_id, p.shift
"""
result4 = pd.read_sql(q4, conn)
print(result4.to_string(index=False))

In [ ]:
# --- SQL Query 5: Subquery — Machines above average temperature ---
print("=== Query 5: Machines Above Average Temperature ===")
q5 = """
SELECT
    s.machine_id,
    m.machine_name,
    ROUND(AVG(s.temperature_c), 2) AS machine_avg_temp,
    ROUND(
        (SELECT AVG(temperature_c) FROM sensor_data), 2
    ) AS fleet_avg_temp
FROM sensor_data s
JOIN machines m ON s.machine_id = m.machine_id
GROUP BY s.machine_id, m.machine_name
HAVING machine_avg_temp > (SELECT AVG(temperature_c) FROM sensor_data)
ORDER BY machine_avg_temp DESC
"""
result5 = pd.read_sql(q5, conn)
print(result5.to_string(index=False))

---
## Part 4: Case Study — Merging Sensor Data with ERP Data

In [ ]:
# ---- Create ERP Dataset ----
n_erp = 200

erp_df = pd.DataFrame({
    "erp_timestamp": [start + timedelta(hours=i) for i in range(0, n_erp * 2, 2)],
    "machine_id": np.tile(machines, n_erp // len(machines)),
    "product_type": np.random.choice(
        ["Printer_A4", "Printer_A3", "Scanner_Pro", "Projector_HD"], n_erp
    ),
    "planned_units": np.random.randint(180, 220, n_erp),
    "material_cost_usd": np.round(np.random.uniform(500, 2000, n_erp), 2),
    "labor_cost_usd": np.round(np.random.uniform(100, 400, n_erp), 2),
    "order_priority": np.random.choice(["High", "Medium", "Low"], n_erp,
                                        p=[0.2, 0.5, 0.3]),
})

erp_df["total_cost_usd"] = erp_df["material_cost_usd"] + erp_df["labor_cost_usd"]

print(f"ERP dataset shape: {erp_df.shape}")
erp_df.head()

In [ ]:
# ---- Sensor data (hourly averages) ----
sensor_hourly = df_treated.copy()
sensor_hourly["timestamp_hour"] = sensor_hourly["timestamp"].dt.floor("H")
sensor_agg = sensor_hourly.groupby(["timestamp_hour", "machine_id"]).agg(
    avg_temperature=("temperature_c", "mean"),
    avg_vibration=("vibration_mms", "mean"),
    avg_pressure=("pressure_hpa", "mean"),
    total_energy=("energy_kwh", "sum"),
    avg_rpm=("rpm", "mean")
).reset_index().round(2)

print(f"Aggregated sensor data: {sensor_agg.shape}")
sensor_agg.head()

In [ ]:
# ---- Merge Sensor + ERP ----
merged_df = pd.merge(
    erp_df,
    sensor_agg,
    left_on=["erp_timestamp", "machine_id"],
    right_on=["timestamp_hour", "machine_id"],
    how="inner"
)

# Derived metrics
merged_df["cost_per_unit"] = np.round(
    merged_df["total_cost_usd"] / merged_df["planned_units"], 2
)
merged_df["energy_per_unit"] = np.round(
    merged_df["total_energy"] / merged_df["planned_units"], 4
)
merged_df["temp_efficiency_flag"] = merged_df["avg_temperature"] > 77

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Matched records: {len(merged_df)} out of {len(erp_df)} ERP records")
merged_df.head()

In [ ]:
# ---- Business Analysis on Merged Data ----

print("=" * 58)
print("  INTEGRATED ANALYSIS: Sensor + ERP Merged Dataset")
print("=" * 58)

# Cost efficiency by product
cost_by_product = merged_df.groupby("product_type")["cost_per_unit"].mean().round(2)
print("\n1. Cost per Unit by Product:")
print(cost_by_product.to_string())

# Energy efficiency by machine
energy_by_machine = merged_df.groupby("machine_id")["energy_per_unit"].mean().round(4)
print("\n2. Energy per Unit by Machine (kWh/unit):")
print(energy_by_machine.to_string())

# High temperature impact on cost
cost_by_temp_flag = merged_df.groupby("temp_efficiency_flag")["cost_per_unit"].mean().round(2)
print("\n3. Cost per Unit — Normal vs High Temperature:")
print(f"   Normal temp (≤77°C): ${cost_by_temp_flag.get(False, 'N/A')}")
print(f"   High temp  (>77°C):  ${cost_by_temp_flag.get(True, 'N/A')}")

# Priority order analysis
priority_analysis = merged_df.groupby("order_priority").agg(
    records=("machine_id", "count"),
    avg_cost=("total_cost_usd", "mean"),
    avg_temp=("avg_temperature", "mean")
).round(2)
print("\n4. Analysis by Order Priority:")
print(priority_analysis.to_string())

In [ ]:
# ---- Integrated Visualization ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Integrated Sensor + ERP Analysis — Epson Operations",
             fontsize=13, fontweight="bold")

# 1. Cost per unit by product type
sns.barplot(data=merged_df, x="product_type", y="cost_per_unit",
            ax=axes[0, 0], palette="muted", estimator=np.mean, ci="sd")
axes[0, 0].set_title("Avg Cost per Unit by Product")
axes[0, 0].set_xlabel("Product Type")
axes[0, 0].set_ylabel("Cost (USD/unit)")
axes[0, 0].tick_params(axis="x", rotation=20)

# 2. Energy per unit by machine
energy_m = merged_df.groupby("machine_id")["energy_per_unit"].mean()
axes[0, 1].bar(energy_m.index, energy_m.values,
               color=["#4C72B0","#55A868","#C44E52","#8172B2"])
axes[0, 1].set_title("Energy Consumption per Unit by Machine")
axes[0, 1].set_ylabel("kWh / unit")

# 3. Temperature vs cost per unit scatter
sc = axes[1, 0].scatter(merged_df["avg_temperature"], merged_df["cost_per_unit"],
                         c=merged_df["total_energy"], cmap="YlOrRd", alpha=0.5, s=20)
plt.colorbar(sc, ax=axes[1, 0], label="Total Energy (kWh)")
axes[1, 0].axvline(77, color="red", linestyle="--", linewidth=0.8, label="Threshold (77°C)")
axes[1, 0].set_title("Temperature vs Cost per Unit")
axes[1, 0].set_xlabel("Avg Temperature (°C)")
axes[1, 0].set_ylabel("Cost (USD/unit)")
axes[1, 0].legend()

# 4. Cost distribution by priority
sns.boxplot(data=merged_df, x="order_priority", y="total_cost_usd",
            order=["High", "Medium", "Low"], ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Total Cost Distribution by Order Priority")
axes[1, 1].set_ylabel("Total Cost (USD)")

plt.tight_layout()
plt.savefig("day4_integrated_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("\n✅ Integrated analysis chart saved as day4_integrated_analysis.png")

---
## Part 5: Complete Data Pipeline Summary

In [ ]:
print("=" * 62)
print("  END-TO-END DATA PIPELINE SUMMARY")
print("=" * 62)

print("""
PIPELINE STAGES:
  1. DATA INGESTION
     ├─ IoT sensors → sensor_data table
     ├─ MES system  → production table
     └─ ERP system  → erp_df

  2. DATA CLEANING
     ├─ Missing values → imputed (median, ffill, interpolation)
     ├─ Duplicates     → removed
     └─ Data types     → standardized

  3. OUTLIER TREATMENT
     └─ Winsorization (IQR bounds) on sensor variables

  4. DATA STORAGE & QUERYING
     └─ SQLite: SELECT, JOIN, GROUP BY, HAVING, Subqueries

  5. DATA INTEGRATION
     └─ Sensor (IoT) × ERP merged on [timestamp, machine_id]

  6. ANALYSIS & INSIGHTS
     ├─ Cost per unit by product and machine
     ├─ Energy efficiency by machine
     ├─ Temperature impact on cost
     └─ Priority-based production analysis
""")

print("KEY FINDINGS:")
cheapest = merged_df.groupby("product_type")["cost_per_unit"].mean().idxmin()
most_efficient_energy = merged_df.groupby("machine_id")["energy_per_unit"].mean().idxmin()
high_temp_cost = merged_df[merged_df["temp_efficiency_flag"]]["cost_per_unit"].mean()
normal_cost = merged_df[~merged_df["temp_efficiency_flag"]]["cost_per_unit"].mean()
cost_impact = (high_temp_cost - normal_cost) / normal_cost * 100

print(f"  ✅ Most cost-efficient product: {cheapest}")
print(f"  ⚡ Most energy-efficient machine: {most_efficient_energy}")
print(f"  🔥 High temperature increases cost by {cost_impact:.1f}%")
print(f"  📊 Total integrated records analyzed: {len(merged_df)}")

In [ ]:
# Close database connection
conn.close()
print("✅ Database connection closed")

---
## ✅ Day 4 Exercises

1. **Missing Values**: In `df_raw`, add another column `humidity_pct` with 12% missing values. Apply an appropriate imputation strategy and justify your choice.
2. **Outliers**: Use the Z-score method (|z| > 3) on `rpm` to detect outliers. How many outliers are found? Remove them instead of capping.
3. **SQL Query**: Write a SQL query to find the TOP 3 machines by total energy consumption, including their machine name from the `machines` table.
4. **SQL Query**: Write a SQL query that returns machines where the maximum temperature exceeded 80°C **AND** the total energy exceeded 5000 kWh.
5. **Data Merge**: Merge the `merged_df` dataset with `df_machines` on `machine_id` to add `machine_type` and `location`. Analyze whether machine type affects energy per unit.